In [1]:
import numpy as np

# GridWorld MDP

grille de 5x5
l'agent commence en haut à gauche,

*   toutes les transitions valent 0 sauf vers états terminaux,
*   état terminal 'en haut à droite', y aller rapporte -3,


*   état terminal 'en bas à droite', y aller rapporte +1,
*   si l'agent se 'cogne contre un mur (faire "gauche" quand on est tout à gauche par ex. , il ne se passe 'rien', l'agent reste sur place et toujours reward de 0









In [14]:
import numpy as np

class GridWorld:
    def __init__(self, rows=5, cols=5, terminal_states=[4, 24], rewards={4: -3.0, 24: 1.0}):
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4  # 0: gauche, 1: droite, 2: haut, 3: bas
        self.terminal_states = terminal_states
        self.rewards_dict = rewards
        self.R = sorted(list(set(rewards.values()) | {0.0}))  # liste des récompenses possibles incluant 0.0
        self.p = np.zeros((self.n_states, self.n_actions, self.n_states, len(self.R)))

        self._build_transitions()

    def _state_to_coords(self, state):
        return state // self.cols, state % self.cols

    def _coords_to_state(self, row, col):
        return row * self.cols + col

    def _build_transitions(self):
        for s in range(self.n_states):
            for a in range(self.n_actions):
                if s in self.terminal_states:
                    # Etat terminal : reste sur place, récompense 0
                    r_idx = self.R.index(0.0)
                    self.p[s, a, s, r_idx] = 1.0
                    continue

                row, col = self._state_to_coords(s)

                # Calcul de la position après action
                if a == 0:  # gauche
                    new_col = max(col - 1, 0)
                    new_row = row
                elif a == 1:  # droite
                    new_col = min(col + 1, self.cols - 1)
                    new_row = row
                elif a == 2:  # haut
                    new_row = max(row - 1, 0)
                    new_col = col
                elif a == 3:  # bas
                    new_row = min(row + 1, self.rows - 1)
                    new_col = col

                s_next = self._coords_to_state(new_row, new_col)

                # Déterminer la récompense pour la transition
                if s_next == s:
                    reward = 0.0  # coupé par un mur, pas de déplacement ni récompense
                elif s_next in self.rewards_dict:
                    reward = self.rewards_dict[s_next]  # récompense terminale
                else:
                    reward = 0.0

                # Trouver l’indice de la récompense dans R
                r_idx = self.R.index(reward)

                # Mettre à jour la probabilité de transition et récompense
                self.p[s, a, s_next, r_idx] = 1.0

    def get_transitions(self):
        # Retourne p 4D et liste des récompenses
        return self.p, self.R

    def is_terminal(self, state):
        return state in self.terminal_states


# Iterative Policy Evaluation



### Explication de l’algorithme d’évaluation itérative de politique

L’algorithme d’évaluation itérative de politique permet d’estimer la **valeur d’un état** $V(s)$ sous une politique donnée $\pi$. Cette valeur correspond à la récompense moyenne cumulée que l’agent peut espérer en partant de cet état et en suivant la politique. L’algorithme procède par des mises à jour successives de $V(s)$ en appliquant la relation de Bellman, qui exprime $V(s)$ en fonction des valeurs des états suivants.

Pour chaque état $s$, la nouvelle valeur est calculée comme la somme pondérée sur toutes les actions possibles $a$, selon la probabilité $\pi(a|s)$ de les choisir, puis sur tous les états suivants $s'$ et toutes les récompenses possibles $r$, pondérée par la probabilité de transition $p(s', r | s, a)$. L’algorithme itère jusqu’à ce que la valeur des états converge, c’est-à-dire que la différence maximale entre deux mises à jour successives soit inférieure à un seuil donné.

Les états terminaux sont considérés comme des points fixes, leur valeur ne change pas, car l’épisode s’arrête lorsqu’on les atteint.

Cette méthode est générique et s’applique à n’importe quel environnement défini par ses états, actions, transitions probabilistes et politique.



In [7]:
from typing import List

In [15]:
def iterative_policy_evaluation(
    pi: np.ndarray,         # politique : matrice (état x action) des probabilités π(a|s)
    S: list,                # liste des états
    A: list,                # liste des actions
    R: list,                # liste des récompenses possibles
    T: list,                # liste des états terminaux
    p: np.ndarray,          # matrice de transition 4D : p[s, a, s', r_index]
    theta: float = 0.0001,  # seuil de convergence
    gamma: float = 0.99,    # facteur de discount
):
    # Initialisation aléatoire des valeurs d'états V(s)
    V = np.random.random((len(S),))
    # Fixer la valeur des états terminaux à 0 (car épisode terminé)
    V[T] = 0.0

    while True:
        delta = 0.0  # variable pour mesurer la plus grande différence entre V ancien et nouveau

        # Parcours de tous les états
        for s in S:
            v = V[s]  # sauvegarde de l'ancienne valeur de l'état s
            total = 0.0  # variable temporaire pour stocker la nouvelle valeur de l'état s

            # Pour chaque action possible dans l'état s
            for a in A:
                sub_total = 0.0  # somme sur tous les états suivants et récompenses

                # Pour chaque état possible suivant s'
                for s_p in S:
                    # Pour chaque récompense possible
                    for r_index in range(len(R)):
                        r = R[r_index]  # récupérer la récompense correspondante
                        # ajouter la contribution pondérée par la probabilité de transition
                        sub_total += p[s, a, s_p, r_index] * (r + gamma * V[s_p])

                # Pondérer par la probabilité d'exécuter l'action a dans l'état s selon la politique
                total += pi[s, a] * sub_total

            V[s] = total  # mise à jour de la valeur de l'état s

            # Calcul de la différence absolue entre l'ancienne et la nouvelle valeur
            abs_diff = abs(v - V[s])
            # Mise à jour de delta avec la différence maximale observée
            delta = max(delta, abs_diff)

        # Condition d'arrêt : si la plus grande mise à jour est inférieure au seuil, on stoppe
        if delta < theta:
            break

    return V  # retourne le vecteur des valeurs d'états estimées


## TEST iterative policy evaluation

In [16]:
# Initialisation de l'environnement
env = GridWorld()

S = list(range(env.n_states))
A = list(range(env.n_actions))
T = env.terminal_states
p, R = env.get_transitions()

# Politique : toujours aller en haut (action 2)
pi_always_up = np.zeros((len(S), len(A)))
pi_always_up[:, 2] = 1.0

V_up = iterative_policy_evaluation(pi_always_up, S, A, R, T, p)
print("Valeurs sous politique always up :")
print(V_up)

# Politique : toujours aller en bas (action 3)
pi_always_down = np.zeros((len(S), len(A)))
pi_always_down[:, 3] = 1.0

V_down = iterative_policy_evaluation(pi_always_down, S, A, R, T, p)
print("\nValeurs sous politique always down :")
print(V_down)

# Politique : toujours aller à droite (action 1)
pi_always_right = np.zeros((len(S), len(A)))
pi_always_right[:, 1] = 1.0

V_right = iterative_policy_evaluation(pi_always_right, S, A, R, T, p)
print("\nValeurs sous politique always right :")
print(V_right)

# Politique : toujours aller à gauche (action 0)
pi_always_left = np.zeros((len(S), len(A)))
pi_always_left[:, 0] = 1.0

V_left = iterative_policy_evaluation(pi_always_left, S, A, R, T, p)
print("\nValeurs sous politique always left :")
print(V_left)

# Politique uniforme aléatoire (toutes actions équiprobables)
pi_uniform = np.ones((len(S), len(A))) / len(A)

V_uniform = iterative_policy_evaluation(pi_uniform, S, A, R, T, p)
print("\nValeurs sous politique uniforme aléatoire :")
print(V_uniform)


Valeurs sous politique always up :
[ 0.00525078  0.00983544  0.00846198  0.00828474  0.          0.00519827
  0.00973708  0.00837736  0.00820189 -3.          0.00514629  0.00963971
  0.00829358  0.00811987 -2.97        0.00509482  0.00954331  0.00821065
  0.00803867 -2.9403      0.00504387  0.00944788  0.00812854  0.00795829
  0.        ]

Valeurs sous politique always down :
[1.46689721e-03 2.18734964e-03 6.00569225e-04 9.89557815e-03
 0.00000000e+00 1.46689721e-03 2.18734964e-03 6.00569225e-04
 9.89557815e-03 9.80100000e-01 1.46689721e-03 2.18734964e-03
 6.00569225e-04 9.89557815e-03 9.90000000e-01 1.46689721e-03
 2.18734964e-03 6.00569225e-04 9.89557815e-03 1.00000000e+00
 1.46689721e-03 2.18734964e-03 6.00569225e-04 9.89557815e-03
 0.00000000e+00]

Valeurs sous politique always right :
[-2.910897   -2.9403     -2.97       -3.          0.          0.00617781
  0.00617781  0.00617781  0.00617781  0.00617781  0.00987845  0.00987845
  0.00987845  0.00987845  0.00987845  0.00487831  0.0



### 1. Politique **always up** (toujours aller en haut)

* Valeurs très faibles, proches de 0, sauf pour les états terminaux avec récompense 0 (en haut à droite = -3) et en bas à droite = 0 (mais tu as -3 à l’état 9 et -2.97 à l’état 14 etc., ça semble cohérent car on s’éloigne des terminaux)
* Cela correspond bien à l’idée que l’agent essaie d’aller vers le haut mais finit souvent bloqué ou avec des récompenses négatives.

---

### 2. Politique **always down** (toujours aller en bas)

* Ici, on voit clairement que les valeurs proches de 1 apparaissent vers la fin (états proches de l’état terminal avec récompense +1).
* Par exemple, état 19, 14, 9 ont des valeurs vers 1, signe que cette politique mène efficacement vers l’état terminal positif en bas à droite (24).
* C’est logique, la politique “toujours bas” est bonne pour aller vers +1.

---

### 3. Politique **always right** (toujours aller à droite)

* On remarque des valeurs négatives très fortes (-3, -2.94...) au début de la grille (états proches du coin en haut à droite), ce qui correspond à la récompense négative en haut à droite.
* Puis vers les états en bas, les valeurs deviennent positives et proches de 1, car on se rapproche de l’autre état terminal positif.
* C’est cohérent avec la grille et ta définition des récompenses.

---

### 4. Politique **always left**

* Les valeurs sont toutes faibles (autour de 0.008) et proches les unes des autres.
* Cela indique que la politique ne mène jamais aux états terminaux, ou que l’agent reste “bloqué” à gauche, sans atteindre ni +1 ni -3.
* Ce comportement est logique car aller toujours à gauche dans une grille en partant de la gauche bloque l’agent (pas de mouvement possible).

---

### 5. Politique **uniforme aléatoire**

* Les valeurs sont majoritairement négatives, décroissantes vers les états terminaux avec récompense négative.
* Cela correspond à un comportement “moyen” où l’agent essaie un peu toutes les directions sans but clair.
* La valeur 0 à l’état terminal (en haut à droite) est cohérente (puisque reward=-3, et état terminal), et la valeur 0.192 à la fin montre un certain espoir d’atteindre le terminal positif.

---

## Conclusion générale

* Tes valeurs sont globalement **cohérentes avec les politiques définies et la structure de la grille**.
* La politique “always down” et “always right” atteignent bien l’état terminal positif (+1), ce qui explique leurs valeurs plus élevées près de ce terminal.
* La politique “always up” conduit vers le terminal négatif (-3), donc valeurs négatives.
* La politique “always left” ne mène pas aux terminaux, donc valeurs très faibles.
* La politique uniforme donne un mix, avec des valeurs négatives dues à la présence du terminal négatif.

# Policy Iteration



### 🔁 **Policy Iteration — Rappel basé sur l’implémentation**

L’algorithme **Policy Iteration** combine deux étapes principales de manière itérative :

#### 1. **Policy Evaluation**

On commence avec une politique initiale aléatoire.
À cette étape, on calcule la valeur de chaque état $V(s)$ en supposant que l’agent suit cette politique.
Cela se fait en résolvant (de façon itérative) l’équation de Bellman pour la valeur d’une politique :

$$
V(s) = \sum_{s', r} p(s, \pi(s), s', r) \cdot \left[ r + \gamma V(s') \right]
$$

On continue jusqu'à convergence, c’est-à-dire jusqu’à ce que les valeurs changent très peu entre deux itérations (moins qu’un seuil `θ`).

#### 2. **Policy Improvement**

Une fois les valeurs $V(s)$ estimées, on met à jour la politique en choisissant, pour chaque état, **l’action qui maximise la valeur attendue** :

$$
\pi_{\text{new}}(s) = \arg\max_a \sum_{s', r} p(s, a, s', r) \cdot \left[ r + \gamma V(s') \right]
$$

Si la politique ne change pas après cette étape, l’algorithme s’arrête : on a trouvé une politique optimale.

---

Cet algorithme est garanti de converger en un nombre fini d’itérations vers une **politique optimale**, tant que l’environnement est fini et le facteur $\gamma < 1$.




In [17]:
def policy_iteration( 
    S: List[int],                # Liste des états
    A: List[int],                # Liste des actions
    R: List[int],                # Liste des récompenses possibles
    T: List[int],                # États terminaux
    p: np.ndarray,               # Matrice de transition (S x A x S' x R)
    theta: float = 0.0001,       # Seuil de convergence pour l’évaluation de politique
    gamma: float = 0.99          # Facteur d’actualisation
):
    V = np.random.random((len(S),))   # Initialisation aléatoire de la valeur des états
    V[T] = 0.0                         # Valeur des états terminaux = 0

    # Politique initiale aléatoire (une action par état)
    pi = np.array([np.random.choice(A) for s in S])
    pi[T] = 0   # Fixe arbitrairement l’action des terminaux (ils ne sont jamais exécutés)

    while True:
        # --- Policy Evaluation ---
        while True:
            delta = 0.0
            for s in S:
                v = V[s]
                total = 0.0
                for s_p in S:
                    for r_index in range(len(R)):
                        r = R[r_index]
                        total += p[s, pi[s], s_p, r_index] * (r + gamma * V[s_p])
                V[s] = total
                delta = max(delta, abs(v - V[s]))  # changement max sur un état
            if delta < theta:
                break  # Stop si convergence atteinte

        # --- Policy Improvement ---
        policy_stable = True
        for s in S:
            old_action = pi[s]
            best_a = None
            best_a_score = -float('inf')
            for a in A:
                score = 0.0
                for s_p in S:
                    for r_index in range(len(R)):
                        r = R[r_index]
                        score += p[s, a, s_p, r_index] * (r + gamma * V[s_p])
                if best_a is None or score > best_a_score:
                    best_a = a
                    best_a_score = score
            if best_a != old_action:
                policy_stable = False
            pi[s] = best_a

        if policy_stable:
            break  # Stop si plus aucun changement de politique

    return pi, V


In [18]:
env = GridWorld()
S = list(range(env.n_states))
A = list(range(env.n_actions))
T = env.terminal_states
p, R = env.get_transitions()

pi_opt, V_opt = policy_iteration(S, A, R, T, p)

print("Politique optimale (action par état) :")
print(pi_opt)

print("\nValeur des états sous cette politique :")
print(V_opt)


Politique optimale (action par état) :
[1 1 1 3 0 1 1 1 1 3 1 1 1 1 3 1 1 1 1 3 1 1 1 1 0]

Valeur des états sous cette politique :
[0.93206535 0.94148015 0.95099005 0.96059601 0.         0.94148015
 0.95099005 0.96059601 0.970299   0.9801     0.95099005 0.96059601
 0.970299   0.9801     0.99       0.96059601 0.970299   0.9801
 0.99       1.         0.970299   0.9801     0.99       1.
 0.        ]




### ✅ **Analyse de la politique optimale (`pi_opt`)**

```python
[1 1 1 3 0
 1 1 1 1 3
 1 1 1 1 3
 1 1 1 1 3
 1 1 1 1 0]
```

🔹 Ce vecteur indique, pour chaque état (de 0 à 24), l'action optimale à prendre :

* `0` = gauche ←
* `1` = droite →
* `2` = haut ↑
* `3` = bas ↓

🔹 Interprétation dans la **grille 5x5** :

|   |   |   |   |       |
| - | - | - | - | ----- |
| → | → | → | ↓ | T(-3) |
| → | → | → | → | ↓     |
| → | → | → | → | ↓     |
| → | → | → | → | ↓     |
| → | → | → | → | T(+1) |

🔸 **Remarques :**

* L’agent se déplace vers la **droite**, puis descend verticalement vers la sortie à récompense **+1** en bas à droite.
* Il **évite l’état terminal à -3** (en haut à droite), ce qui est un comportement intelligent et optimal.
* La dernière action `0` à l’état 24 est arbitraire, car c’est un **état terminal**.

---

### 📈 **Analyse des valeurs d’état (`V_opt`)**

```python
[0.93  0.94  0.95  0.96  0.00
 0.94  0.95  0.96  0.97  0.98
 0.95  0.96  0.97  0.98  0.99
 0.96  0.97  0.98  0.99  1.00
 0.97  0.98  0.99  1.00  0.00]
```

🔹 Ces valeurs montrent que :

* Plus on se rapproche de **l’état terminal positif (24)**, plus la valeur est haute (≈1).
* L’état terminal négatif (état 4) a une valeur de 0, car aucune récompense n’est obtenue après son entrée.
* Les états menant vers l’état 4 sont **évités naturellement**, car leurs valeurs sont inférieures à ceux qui mènent à l’état 24.

---

###  **Conclusion à mettre dans ton rapport**

> La politique optimale trouvée par `Policy Iteration` dirige l’agent vers l’état terminal situé en bas à droite (état 24), qui donne une récompense positive de +1.
> En revanche, elle évite l’état terminal en haut à droite (état 4), qui donne une récompense négative de -3.
> Cela montre que l’algorithme a correctement appris à maximiser les récompenses à long terme, en exploitant la structure de l’environnement.




# Value iteration


L’algorithme de **Value Iteration** est une méthode fondamentale en apprentissage par renforcement utilisée pour calculer la fonction de valeur optimale V∗V^* et en déduire la politique optimale π∗\pi^*.

### Fonctionnement de l’algorithme (basé sur mon code)

- On initialise la fonction de valeur V(s)V(s) à zéro pour tous les états ss, en fixant explicitement V(s)=0V(s) = 0 pour les états terminaux.
    
- On itère ensuite jusqu’à convergence :
    
    - Pour chaque état non terminal, on met à jour la valeur V(s)V(s) en prenant le maximum attendu sur toutes les actions possibles, c’est-à-dire :
        
    
    V(s)←max⁡a∑s′,rp(s,a,s′,r)×(r+γV(s′))V(s) \leftarrow \max_a \sum_{s', r} p(s, a, s', r) \times \big(r + \gamma V(s')\big)
    
    où p(s,a,s′,r)p(s,a,s',r) est la probabilité de transition vers l’état s′s' avec récompense rr en prenant l’action aa depuis ss, et γ\gamma est le facteur d’actualisation.
    
- La mise à jour se fait jusqu’à ce que la différence maximale entre deux itérations successives de VV soit inférieure à un seuil θ\theta fixé (critère de convergence).
    
- Après convergence de VV, on extrait la politique optimale en choisissant pour chaque état l’action maximisant l’espérance de la récompense cumulée future.

In [19]:
from typing import List
import numpy as np

def value_iteration(
    S: List[int],       # Liste des états
    A: List[int],       # Liste des actions possibles
    R: List[int],       # Liste des récompenses possibles
    T: List[int],       # États terminaux
    p: np.ndarray,      # Matrice de transition de forme (S, A, S', R)
    theta: float = 0.0001,  # Seuil pour arrêter l’itération (précision)
    gamma: float = 0.99     # Facteur de réduction (discount factor)
):
    # Initialisation des valeurs des états à 0
    V = np.zeros(len(S))
    V[T] = 0.0  # On fixe les valeurs des états terminaux à 0

    # Phase d’itération de Bellman : mise à jour des valeurs jusqu’à convergence
    while True:
        delta = 0.0  # Suivi de la plus grande variation sur une itération

        for s in S:
            if s in T:
                continue  # On ne met pas à jour les états terminaux

            v = V[s]  # Valeur actuelle de l’état s

            # Pour chaque action, on calcule la valeur espérée et on garde le maximum
            V[s] = max(
                sum(
                    p[s, a, s_p, r_index] * (R[r_index] + gamma * V[s_p])
                    for s_p in S
                    for r_index in range(len(R))
                )
                for a in A
            )

            # On garde la plus grande variation pour voir si on a convergé
            delta = max(delta, abs(v - V[s]))

        # Si les valeurs changent très peu, on s’arrête
        if delta < theta:
            break

    # Politique optimale : on choisit la meilleure action dans chaque état
    pi = np.zeros(len(S), dtype=int)  # Politique initiale (vecteur d’actions)

    for s in S:
        if s in T:
            continue  # Pas de politique à apprendre pour un état terminal

        best_a = None
        best_value = -np.inf  # Valeur initiale très basse

        # On cherche l’action qui donne la meilleure valeur attendue
        for a in A:
            value = sum(
                p[s, a, s_p, r_index] * (R[r_index] + gamma * V[s_p])
                for s_p in S
                for r_index in range(len(R))
            )
            if value > best_value:
                best_value = value
                best_a = a

        pi[s] = best_a  # Meilleure action choisie pour l’état s

    return pi, V  # Retourne la politique et la fonction de valeur optimales


In [20]:
# Initialisation de l'environnement GridWorld
env = GridWorld()
S = list(range(env.n_states))         # États
A = list(range(env.n_actions))        # Actions
T = env.terminal_states               # États terminaux
p, R = env.get_transitions()         # Matrices de transitions et récompenses

# Exécution de l'algorithme de value iteration
pi_vi, V_vi = value_iteration(S, A, R, T, p)

# Affichage des résultats
print("Politique optimale (issue de value iteration) :")
print(pi_vi)

print("\nValeur des états (issue de value iteration) :")
print(V_vi)


Politique optimale (issue de value iteration) :
[1 1 1 3 0 1 1 1 1 3 1 1 1 1 3 1 1 1 1 3 1 1 1 1 0]

Valeur des états (issue de value iteration) :
[0.93206535 0.94148015 0.95099005 0.96059601 0.         0.94148015
 0.95099005 0.96059601 0.970299   0.9801     0.95099005 0.96059601
 0.970299   0.9801     0.99       0.96059601 0.970299   0.9801
 0.99       1.         0.970299   0.9801     0.99       1.
 0.        ]




### ✅ **Politiques optimales obtenues :**

```python
[1 1 1 3 0 
 1 1 1 1 3 
 1 1 1 1 3 
 1 1 1 1 3 
 1 1 1 1 0]
```

---

### ✅ **Fonction de valeur optimale :**

```python
[0.93206535 0.94148015 0.95099005 0.96059601 0.         
 0.94148015 0.95099005 0.96059601 0.970299   0.9801     
 0.95099005 0.96059601 0.970299   0.9801     0.99       
 0.96059601 0.970299   0.9801     0.99       1.         
 0.970299   0.9801     0.99       1.         0.        ]
```

---

### 🔍 **Analyse des résultats (à mettre dans ton rapport) :**

* On observe que **la politique optimale issue de Value Iteration est exactement la même que celle obtenue avec Policy Iteration**, ce qui est une excellente nouvelle : les deux algorithmes convergent bien vers la même solution.
* Les **valeurs maximales sont proches de 1** dans les états situés en bas à droite de la grille, ce qui est cohérent car l’agent reçoit une **récompense de +1** en atteignant le terminal bas-droit.
* Les **valeurs décroissent progressivement** à mesure qu’on s’éloigne de cet état terminal positif, ce qui est logique avec un facteur de discount γ = 0.99.
* L’action `1` correspond à **"droite"** et `3` à **"bas"**, ce qui confirme que l’agent essaie toujours d’avancer vers le **coin bas droit** où il reçoit la récompense.
* L’état 4 (coin haut-droit, avec récompense -3) est évité : la politique optimale **ne choisit pas de s’y diriger**.

---

###  Conclusion :

> L'algorithme **Value Iteration** converge vers la même politique optimale que **Policy Iteration**, ce qui confirme la justesse de son implémentation. Les valeurs d'états reflètent clairement la distance aux états terminaux, avec des valeurs croissantes vers le terminal positif (+1) et une politique qui évite le terminal négatif (-3).
> Ce résultat valide le fonctionnement correct et générique de l’algorithme, capable de s’adapter automatiquement à la dynamique de l’environnement sans ajustement spécifique.




# GridWorld_Monte Carlo

In [21]:
from tqdm import tqdm
import numpy as np

In [22]:
class MonteCarloEnv:

  def num_states(self) -> int:
    raise NotImplementedError()

  def num_actions(self) -> int:
    raise NotImplementedError()

  def step(self, a: int):
    raise NotImplementedError()

  def score(self) -> float:
    raise NotImplementedError()

  def is_game_over(self) -> bool:
    raise NotImplementedError()

  def reset(self):
    raise NotImplementedError()

In [24]:
class GridWorld_MC(MonteCarloEnv):
    def __init__(self):
        self.s = 0  # position initiale (état)
        self.inner_score = 0.0

    def num_states(self) -> int:
        return 25

    def num_actions(self) -> int:
        return 4  # 0: gauche, 1: droite, 2: haut, 3: bas

    def state(self) -> int:
        return self.s

    def step(self, a: int):
        if self.is_game_over():
            raise Exception("Épisode terminé")

        row, col = divmod(self.s, 5)

        if a == 0 and col > 0:  # gauche
            col -= 1
        elif a == 1 and col < 4:  # droite
            col += 1
        elif a == 2 and row > 0:  # haut
            row -= 1
        elif a == 3 and row < 4:  # bas
            row += 1

        new_state = row * 5 + col

        # Calcul de la récompense immédiate
        if new_state == 4:
            reward = -3.0
        elif new_state == 24:
            reward = 1.0
        else:
            reward = 0.0

        self.s = new_state
        self.inner_score += reward  # si tu souhaites garder score cumulé

        done = self.is_game_over()
        return new_state, reward, done

    def score(self) -> float:
        return self.inner_score

    def is_game_over(self) -> bool:
        return self.s == 4 or self.s == 24

    def reset(self):
        self.s = 0
        self.inner_score = 0.0



##  Étapes de Monte Carlo Exploring Starts

1. **Démarrage aléatoire**
   → On commence chaque épisode avec une **paire (état, action)** choisie **aléatoirement**. Cela garantit l’**exploration de tout l’espace** (tous les couples possibles).

2. **Génération d’un épisode**
   → Une fois la première action jouée, on suit une **politique donnée** (initialement aléatoire ou gloutonne) jusqu’à **atteindre un état terminal**.

3. **Calcul du retour cumulé**
   → Pour chaque paire $(s_t, a_t)$ de l’épisode, on calcule le **retour total** à partir de ce moment-là :

   $$
   G_t = r_{t+1} + \gamma r_{t+2} + \dots
   $$

4. **Mise à jour de la valeur de chaque paire (état, action)**
   → On met à jour $Q(s,a)$ en utilisant la **moyenne des retours observés** pour chaque $(s,a)$.

5. **Amélioration de la politique**
   → On améliore la politique $\pi$ en choisissant, pour chaque état $s$, l’action $a$ qui **maximise la valeur estimée** :

   $$
   \pi(s) = \arg\max_a Q(s, a)
   $$



In [ ]:
import numpy as np
from tqdm import tqdm

def monte_carlo_es(env, episodes_count=100_000, gamma=0.99):
    Q = np.zeros((env.num_states(), env.num_actions()))
    N = np.zeros((env.num_states(), env.num_actions()))  # compteur pour la moyenne incrémentale

    for _ in tqdm(range(episodes_count), desc="Monte Carlo ES"):
        # -------- Exploring Start avec état NON terminal --------
        while True:
            s0 = np.random.randint(env.num_states())
            if not env.is_game_over() and s0 not in (4, 24):  # sécurité renforcée
                break
        a0 = np.random.randint(env.num_actions())

        env.reset()
        env.s = s0
        episode = []
        if env.is_game_over():
            continue

        prev_score = env.score()
        env.step(a0)
        r = env.score() - prev_score
        episode.append((s0, a0, r))

        while not env.is_game_over():
            s = env.state()
            a = np.random.randint(env.num_actions())
            prev_score = env.score()
            env.step(a)
            r = env.score() - prev_score
            episode.append((s, a, r))

        # -------- Calcul du retour G et mise à jour incrémentale --------
        G = 0.0
        visited = set()
        for (s, a, r) in reversed(episode):
            G = r + gamma * G
            if (s, a) not in visited:
                N[s][a] += 1
                Q[s][a] += (G - Q[s][a]) / N[s][a]  # mise à jour incrémentale
                visited.add((s, a))

    pi = np.argmax(Q, axis=1)
    return pi, Q


In [25]:
import numpy as np
from tqdm import tqdm

def monte_carlo_es(env, episodes_count=100_000, gamma=0.99):
    Q = np.zeros((env.num_states(), env.num_actions()))
    returns = [[[] for _ in range(env.num_actions())] for _ in range(env.num_states())]

    for _ in tqdm(range(episodes_count), desc="Monte Carlo ES"):
        # -------- Exploring start avec état NON terminal --------
        while True:
            s0 = np.random.randint(env.num_states())
            if s0 not in (4, 24):  # Éviter les états terminaux
                break
        a0 = np.random.randint(env.num_actions())

        env.reset()
        env.s = s0
        episode = []
        if env.is_game_over():
            continue  # sécurité en plus

        prev_score = env.score()
        env.step(a0)
        r = env.score() - prev_score
        episode.append((s0, a0, r))

        while not env.is_game_over():
            s = env.state()
            a = np.random.randint(env.num_actions())
            prev_score = env.score()
            env.step(a)
            r = env.score() - prev_score
            episode.append((s, a, r))

        # -------- Retour G et mise à jour Q --------
        G = 0.0
        visited = set()
        for (s, a, r) in reversed(episode):
            G = r + gamma * G
            if (s, a) not in visited:
                returns[s][a].append(G)
                Q[s][a] = np.mean(returns[s][a])
                visited.add((s, a))

    pi = np.argmax(Q, axis=1)
    return pi, Q


In [26]:
env = GridWorld_MC()  # utilise la version adaptée Monte Carlo

pi_mc, Q_mc = monte_carlo_es(env, episodes_count=100_000)

print("Politique optimale estimée par Monte Carlo (action par état) :")
print(pi_mc)

print("\nValeurs estimées Q(s,a) :")
print(Q_mc)



Monte Carlo ES: 100%|██████████| 100000/100000 [3:10:15<00:00,  8.76it/s]     

Politique optimale estimée par Monte Carlo (action par état) :
[3 0 0 0 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 0]

Valeurs estimées Q(s,a) :
[[-0.94724894 -1.09957323 -0.945159   -0.86083933]
 [-0.94843753 -1.38908154 -1.07675424 -0.95720788]
 [-1.06270122 -1.94088796 -1.33531011 -1.11940423]
 [-1.32286826 -3.         -1.90116224 -1.37720696]
 [ 0.          0.          0.          0.        ]
 [-0.8666116  -0.95494401 -0.97517445 -0.72769468]
 [-0.87340481 -1.1367448  -1.09833985 -0.7545236 ]
 [-0.94171084 -1.40660746 -1.37350126 -0.76629863]
 [-1.09127205 -1.75342544 -1.9225579  -0.80062247]
 [-1.36916207 -1.72677071 -3.         -0.85249337]
 [-0.71535949 -0.75726436 -0.88588969 -0.57865301]
 [-0.72391291 -0.79157883 -0.97160493 -0.53586064]
 [-0.73782975 -0.81123878 -1.11112522 -0.43959039]
 [-0.73895453 -0.84612145 -1.38660279 -0.24406392]
 [-0.7940772  -0.85054292 -1.75460324 -0.03128849]
 [-0.57009878 -0.53899081 -0.73504703 -0.48278775]
 [-0.57526039 -0.44847099 -0.74106084 -0.4

In [ ]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

def monte_carlo_es_with_convergence(env, episodes_count=100_000, gamma=0.99):
    Q = np.zeros((env.num_states(), env.num_actions()))
    returns = [[[] for _ in range(env.num_actions())] for _ in range(env.num_states())]
    convergence = []  # Liste pour stocker la variation moyenne à chaque épisode

    for episode_num in tqdm(range(episodes_count), desc="Monte Carlo ES"):
        # -------- Exploring start avec état NON terminal --------
        while True:
            s0 = np.random.randint(env.num_states())
            if s0 not in (4, 24):  # éviter états terminaux
                break
        a0 = np.random.randint(env.num_actions())

        env.reset()
        env.s = s0
        episode = []
        if env.is_game_over():
            continue

        prev_score = env.score()
        env.step(a0)
        r = env.score() - prev_score
        episode.append((s0, a0, r))

        while not env.is_game_over():
            s = env.state()
            a = np.random.randint(env.num_actions())
            prev_score = env.score()
            env.step(a)
            r = env.score() - prev_score
            episode.append((s, a, r))

        # Sauvegarder Q avant mise à jour pour mesurer la différence
        Q_old = Q.copy()

        # -------- Retour G et mise à jour Q --------
        G = 0.0
        visited = set()
        for (s, a, r) in reversed(episode):
            G = r + gamma * G
            if (s, a) not in visited:
                returns[s][a].append(G)
                Q[s][a] = np.mean(returns[s][a])
                visited.add((s, a))

        # Calculer la moyenne des différences entre Q et Q_old
        diff = np.abs(Q - Q_old).mean()
        convergence.append(diff)

    pi = np.argmax(Q, axis=1)

    # Tracer la convergence
    plt.figure(figsize=(10, 5))
    plt.plot(convergence)
    plt.yscale('log')  # échelle logarithmique pour mieux voir la décroissance
    plt.xlabel("Épisodes")
    plt.ylabel("Moyenne des changements de Q")
    plt.title("Convergence de Monte Carlo Exploring Starts")
    plt.grid(True)
    plt.show()

    return pi, Q


In [ ]:
env = GridWorld_MC()  # ta classe GridWorld adaptée MonteCarlo
pi_opt, Q_opt = monte_carlo_es_with_convergence(env, episodes_count=100000, gamma=0.99)

print("Politique optimale estimée :")
print(pi_opt)


## QLearning


In [27]:
from tqdm import tqdm
import numpy as np

def q_learning(
    env,                         # environnement (ex : ton gridworld)
    episodes_count=1_000_000,    # Nombre total d'épisodes à exécuter (plus il est grand, plus l'agent a de chances d'apprendre une bonne politique)
    epsilon=1.0,                 # taux d'exploration (1 = 100% d'exploration au début)
    gamma=0.9999,                # Facteur de réduction (discount factor) : plus il est proche de 1, plus on valorise les récompenses futures
    alpha=0.1                    # taux d’apprentissage (vitesse de mise à jour des Q-valeurs)
):
    # Initialisation de la table Q avec des valeurs aléatoires entre -1 et 1
    # Cela évite un biais vers des Q-valeurs nulles au début (meilleure exploration)
    Q = np.random.uniform(-1.0, 1.0, (env.num_states(), env.num_actions()))

    # Boucle principale sur le nombre d’épisodes spécifié
    for ep_id in tqdm(range(episodes_count), desc="Q-Learning in progress"):
        env.reset()              # Réinitialise l’environnement à l’état de départ
        s = env.state()          # Récupère l’état courant (état initial)

        # Boucle pour un seul épisode jusqu’à atteindre un état terminal
        while not env.is_game_over():
            # Choix de l’action à faire dans l’état courant s :
            # - soit aléatoire (exploration, avec proba epsilon)
            # - soit l’action ayant la plus grande Q-valeur (exploitation)
            if np.random.uniform(0.0, 1.0) <= epsilon:
                a = np.random.randint(env.num_actions())  # Action aléatoire
            else:
                a = np.argmax(Q[s])                       # Action optimale actuelle

            # Récupère le score actuel pour calculer la récompense après action
            prev_score = env.score()

            # Effectue l’action choisie dans l’environnement
            env.step(a)

            # La récompense r est définie comme la variation du score
            r = env.score() - prev_score

            # Nouvel état après l’action
            s_p = env.state()

            # Si l’état suivant est terminal, alors toutes ses Q-valeurs sont nulles
            if env.is_game_over():
                Q[s_p, :] = 0.0

            # Mise à jour de la Q-valeur de l’état-action courant (formule Q-learning)
            # Q(s,a) ← Q(s,a) + α × (r + γ × max_a' Q(s',a') − Q(s,a))
            Q[s, a] = Q[s, a] + alpha * (r + gamma * np.max(Q[s_p]) - Q[s, a])

            # Mise à jour de l’état courant (on passe à l’état suivant)
            s = s_p

    # Politique optimale extraite de Q : pour chaque état, action ayant la plus grande Q-valeur
    pi = np.argmax(Q, axis=1)

    return pi, Q  # Renvoie la politique optimale apprise et la Q-table finale


In [28]:
# Instanciation de l'environnement Monte Carlo GridWorld
env = GridWorld_MC()

# Appel de la fonction q_learning
pi_opt, Q_opt = q_learning(env, episodes_count=100_000, epsilon=0.1, gamma=0.99, alpha=0.1)

# Affichage de la politique optimale apprise (action par état)
print("Politique optimale apprise par Q-learning (action par état) :")
print(pi_opt)

# Affichage des valeurs Q finales
print("\nValeurs Q finales (Q-table) :")
print(Q_opt)


Q-Learning in progress: 100%|██████████| 100000/100000 [00:09<00:00, 11012.83it/s]

Politique optimale apprise par Q-learning (action par état) :
[1 1 1 3 0 1 1 1 3 3 1 1 1 1 3 1 1 1 1 3 1 1 1 1 0]

Valeurs Q finales (Q-table) :
[[ 0.92274469  0.93206535  0.92274469  0.93206535]
 [ 0.92274469  0.94148015  0.93206535  0.94148015]
 [ 0.93206535  0.95099005  0.94148015  0.95099005]
 [ 0.94148015 -3.          0.95099005  0.96059601]
 [ 0.          0.          0.          0.        ]
 [ 0.93206535  0.94148015  0.92274469  0.94148015]
 [ 0.93206535  0.95099005  0.93206535  0.95099005]
 [ 0.94148015  0.96059601  0.94148015  0.96059601]
 [ 0.95099005  0.95353695  0.95099005  0.970299  ]
 [ 0.96059601  0.93794267 -2.93811082  0.97213136]
 [ 0.94148015  0.95099005  0.93206535  0.95099005]
 [ 0.94148015  0.96059601  0.94148015  0.96059601]
 [ 0.95099005  0.970299    0.95099005  0.970299  ]
 [ 0.96059601  0.9801      0.96059601  0.9801    ]
 [ 0.970299    0.9801      0.95297629  0.99      ]
 [ 0.95099005  0.96059601  0.94148015  0.96059601]
 [ 0.95099005  0.970299    0.95099005  

In [ ]:
import time
now = time.time()
pi,Q= q_learning(GridWorld())
print(pi)
print(Q)
print(time.time() - now)

Q-Learning in progress: 100%|██████████| 1000000/1000000 [12:24<00:00, 1342.97it/s]

[1 1 1 3 0 1 1 1 1 3 1 1 1 1 3 1 1 1 1 3 1 1 1 1 0]
[[ 0.99920028  0.99930021  0.99920028  0.99930021]
 [ 0.99920028  0.99940015  0.99930021  0.99940015]
 [ 0.99930021  0.9995001   0.99940015  0.9995001 ]
 [ 0.99940015 -3.          0.9995001   0.99960006]
 [ 0.          0.          0.          0.        ]
 [ 0.99930021  0.99940015  0.99920028  0.99940015]
 [ 0.99930021  0.9995001   0.99930021  0.9995001 ]
 [ 0.99940015  0.99960006  0.99940015  0.99960006]
 [ 0.9995001   0.99970003  0.9995001   0.99970003]
 [ 0.99960006  0.99970003 -3.          0.99980001]
 [ 0.99940015  0.9995001   0.99930021  0.9995001 ]
 [ 0.99940015  0.99960006  0.99940015  0.99960006]
 [ 0.9995001   0.99970003  0.9995001   0.99970003]
 [ 0.99960006  0.99980001  0.99960006  0.99980001]
 [ 0.99970003  0.99980001  0.99970003  0.9999    ]
 [ 0.9995001   0.99960006  0.99940015  0.99960006]
 [ 0.9995001   0.99970003  0.9995001   0.99970003]
 [ 0.99960006  0.99980001  0.99960006  0.99980001]
 [ 0.99970003  0.9999      0.9